## Welcome to the basic neural network tutorial!
The following notebook is designed to walk through the process of building and training your first scikit based algorithm using the standalone MaCh3 python utilities!

If you're writing scripts this is encapsulate in the objects in `MaCh3PythonUtils/config_reader` however this guide aims to break apart the process of writing the code for yourself!

The first step is to load in a MaCh3 MCMC fit as input file. This is done using the ChainHandler class!

In [ ]:
# MaCh3Python Deps
from MaCh3PythonUtils.file_handling.chain_handler import ChainHandler
from MaCh3PythonUtils.machine_learning.ml_factory import MLFactory
from MaCh3PythonUtils.file_handling.chain_diagnostics import ChainDiagnostics
from sklearn.model_selection import LearningCurveDisplay
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import log_loss    


# Other imports
from matplotlib import pyplot as plt
from pathlib import Path
import numpy as np
import random as rand
import os
import pandas as pd
import seaborn as sns
import gdown

In [ ]:

#file_url="https://drive.google.com/file/d/1iE6xFhn3BH_HnLUfQ7KFGy2wfeH52Rwf/view?usp=sharing"


path_NoAd_T2K= Path("../models/demo_files 2/NoAdapt/T2K")
path_NoAd_Nova= Path('../models/demo_files 2/NoAdapt/NOvA')
path_Ad_T2K= Path('../models/demo_files 2/Adapt/T2K')
path_Ad_Nova= Path('../models/demo_files 2/Adapt/NOvA')

NAD_T2K=[f"{path_NoAd_T2K}/{x}" for x in os.listdir(path_NoAd_T2K)] 
NAD_Nova=[f'{path_NoAd_Nova}/{y}' for y in os.listdir(path_NoAd_Nova)]
AD_T2K=[f'{path_Ad_T2K}/{z}' for z in os.listdir(path_Ad_T2K)]
AD_Nova=[f'{path_Ad_Nova}/{a}' for a in os.listdir(path_Ad_Nova)]# these will be used later as args in functions (used to calc the dims of the data)

print(NAD_T2K)
print(NAD_Nova)
print(AD_T2K)
print(AD_Nova)


fitting_label = "ID"
paths = [NAD_T2K, NAD_Nova, AD_T2K, AD_Nova]
# chain_list [for file in folder ChainHandler(input_file, chain_name, verbose=verbse)]: Handler List : Hlist

TEMP_NAD_T2K_list = []
TEMP_NAD_Nova_list = []
TEMP_AD_T2K_list = []
TEMP_AD_Nova_list = []

for lm in paths:
    for x in lm:
        chain_handler = ChainHandler(x, 'posteriors', verbose='verbose')
            
        # Process all the files
        chain_handler.ignore_plots(["xsec_20", "xsec_19"])
        chain_handler.add_additional_plots(["xsec_0", "xsec_1", "xsec_2", "xsec_3", "xsec_4", "xsec_5", "xsec_6", "xsec_7", "xsec_8", "xsec_9", "xsec_10", "xsec_11", "xsec_12", "xsec_13",
                                            "xsec_14", "xsec_15", "xsec_16", "xsec_17", "xsec_18", "xsec_21", "xsec_22"])
        #We need to make sure the chain handler knows this exists, passing true means it is looking for some with that exact name
        # chain_handler.add_additional_plots(fitting_label, True)

        # Finally we can do some cuts to get rid of things like burn-in
        chain_handler.add_new_cuts(["LogL<22.5", "step>10000", "delm2_23>0"])
        chain_handler.add_new_cuts(["step>0", "LogL_systematic_xsec_cov<12345"])

        # Last step is  convert the chain data files into a pandas dataframe
        y= chain_handler.convert_ttree_to_array()
        #chain_list_2.append(chain_handler)


        if lm == NAD_T2K:
            TEMP_NAD_T2K_list.append(chain_handler)
        elif lm == NAD_Nova:
            TEMP_NAD_Nova_list.append(chain_handler)
        elif lm == AD_T2K:
            TEMP_AD_T2K_list.append(chain_handler)
        elif lm == AD_Nova:
            TEMP_AD_Nova_list.append(chain_handler)


NAD_T2K_data = TEMP_NAD_T2K_list[0]
NAD_T2K_data.ttree_array['ID']= 0 #give the first file, containing chain data, the chain ID of 0

NAD_Nova_data = TEMP_NAD_Nova_list[0]
NAD_Nova_data.ttree_array['ID']= 0

AD_T2K_data = TEMP_AD_T2K_list[0]
AD_T2K_data.ttree_array['ID']= 0

AD_Nova_data = TEMP_AD_Nova_list[0]
AD_Nova_data.ttree_array['ID']= 0


for id, chain in enumerate(TEMP_NAD_T2K_list[1:], start = 1): #iterate over the rest of the files and give it an acsending order Chain Ids starting from 1
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    #chain.ttree_array[f'xsec_{rand.randint(0,20)}']= np.random.random(size=len(chain.ttree_array))*1.2#on a random selection of parameters, change each value of that entire coloumn from a random float between 0 and 1
    NAD_T2K_data.ttree_array = pd.concat([NAD_T2K_data.ttree_array, chain.ttree_array])

for ida, chains in enumerate(TEMP_NAD_Nova_list[1:], start = 1): 
    #add ID to chain['id']
    chains.ttree_array['ID'] = ida
    #chains.ttree_array[f'xsec_{rand.randint(0,20)}']= np.random.random(size=len(chains.ttree_array))*1.2
    NAD_Nova_data.ttree_array = pd.concat([NAD_Nova_data.ttree_array, chains.ttree_array])

for idb, chainss in enumerate(TEMP_AD_T2K_list[1:], start = 1): 
    #add ID to chain['id']
    chainss.ttree_array['ID'] = idb
    #chainss.ttree_array[f'xsec_{rand.randint(0,20)}']= np.random.random(size=len(chainss.ttree_array))*1.2
    AD_T2K_data.ttree_array = pd.concat([AD_T2K_data.ttree_array, chainss.ttree_array])

for idc, chainsss in enumerate(TEMP_AD_Nova_list[1:], start = 1): 
    #add ID to chain['id']
    chainsss.ttree_array['ID'] = idc
    #chainsss.ttree_array[f'xsec_{rand.randint(0,20)}']= np.random.random(size=len(chainsss.ttree_array))*1.2
    AD_Nova_data.ttree_array = pd.concat([AD_Nova_data.ttree_array, chainsss.ttree_array])

#from now on, the files that include out data that are now usable are: NAD_T2K_data, NAD_Nova_data, AD_T2K_data, AD_Nova_data


## Machine learning
Okay now we've done some very basic file manipulation, it's time for some machine learning!
All algorithms use the same common `MLFactory` interface so let's go about configuring it!

In [ ]:
'''
print(NAD_T2K_data.ttree_array['ID'])

hist = NAD_T2K_data.ttree_array['delta_cp'][NAD_T2K_data.ttree_array['ID']==0].hist()
plt.show()
hist = NAD_T2K_data.ttree_array['delta_cp'][NAD_T2K_data.ttree_array['ID']==1] 

NAD_T2K_data.ttree_array.columns
'''

In [ ]:
# First step we need to initialise the factory
path_NoAd_T2K.parent.mkdir(parents=True, exist_ok=True)# Make sure the model output directory exists
print(fitting_label)
# The factory produces ML models, we need to pass it the chain handler and the fitting label to get started
ml_factory_NADT2K = MLFactory(NAD_T2K_data, fitting_label, f"{path_NoAd_T2K}.pdf")


path_NoAd_Nova.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_NADNova = MLFactory(NAD_Nova_data, fitting_label, f"{path_NoAd_Nova}.pdf")



path_Ad_T2K.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_ADT2K = MLFactory(AD_T2K_data, fitting_label, f"{path_Ad_T2K}.pdf")


path_Ad_Nova.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_ADNova = MLFactory(AD_Nova_data, fitting_label, f"{path_Ad_Nova}.pdf")



Now we need to define a model, for this demonstration we'll make a very simple BDT!

In [ ]:



ml_model_NADT2K = ml_factory_NADT2K.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0868, max_iter=50, verbose=2, max_leaf_nodes=410, 
                                                   l2_regularization= 0.033, early_stopping=True, n_iter_no_change= 10)
ml_model_NADT2K.set_training_test_set(0.2)

'''
ml_model_NADNova = ml_factory_NADNova.make_interface('SciKit', 'histboostclass', loss='log_loss', max_iter=50, verbose=2, max_leaf_nodes=30)
ml_model_NADNova.set_training_test_set(0.3)

ml_model_ADT2K = ml_factory_ADT2K.make_interface('SciKit', 'histboostclass', loss='log_loss', max_iter=20000, verbose=1, max_leaf_nodes=250)
ml_model_ADT2K.set_training_test_set(0.3)

ml_model_ADNova = ml_factory_ADNova.make_interface('SciKit', 'histboostclass', loss='log_loss', max_iter=20000, verbose=1, max_leaf_nodes=250)
ml_model_ADNova.set_training_test_set(0.3)
'''

In [ ]:
#training the model
ml_model_NADT2K.train_model()

scikitmodel_NADT2K= ml_model_NADT2K.model #assigns the model (histboostclass) to a variable; able to access and use the model directly 
ml_model_NADT2K.test_model_class(NAD_T2K, 'noadapt', 't2k', norm= False, model=scikitmodel_NADT2K)#test function 


'''
scikitmodel_NADNova= ml_model_NADNova.model
scikitmodel_ADT2K= ml_model_ADT2K.model
scikitmodel_NADNova= ml_model_ADNova.model
'''


#ml_model_NADT2K.plot_training_loss(scikitmodel_NADT2K)# plots the loss function 



#LearningCurveDisplay.from_estimator( #prints out how well model learns when you give it a training size of 50000, 100000 etx (out of 1 million)
#   scikitmodel_NADT2K, ml_model_NADT2K._training_data, ml_model_NADT2K._training_labels, train_sizes=[50000, 100000, 900000], cv=2) 


## Adaptive versus Non-Adaptive: Visual Comparison
The `ChainDiagnostics` class is used here to plot how an MCMC looks like from a chosen dataset and parameter. 

The plots on the left are called **Trace Plots**. 

> Trace plots visually represent the evolution of parameter values over iterations of Markov Chains.

The x-axis is the step number of the MCMC (note that each tick is that value multiplied by 1e6), and the y-axis shows the actual physical value of a propery of the neutrino. Trace plots help asses how well-mixed the chains are by seeing whether the chains have explored the target distribution adequately.

One can see that the main difference are that the Adaptive MCMC travels through more of the posterioir than the non-adaptive; the Adaptive MCMC oscillates around `1.0` more times per step than the Non-Adaptive.

> The plots on the right are **Histogram** plots and it is just the Trace plots integrated. 

The number of times the MCMC grabs a particular value is then plotted as the Histogram plot. From the Trace plot, one can deduce that the chain "crosses" `1.0` the most, thus the peak of the Hist plot is at `1.0`. 

<div class="alert alert-block alert-success">
<b>Note:</b> One might expect that for different algorithms (i.e non-adaptive versus adaptive), both trace plots and the histogram plots should look different. However, the non-adaptive and adaptive MCMCs are based off of the same physical system, thus the only difference is their algorithm, which is distinguished in the trace plot. Histogram plots are usually identify general trends that are present in a physical systems.

In [ ]:
plotter_NADT2K=ChainDiagnostics(NAD_T2K_data)
plotter_ADT2K=ChainDiagnostics(AD_T2K_data)

plotter_NADT2K("xsec_10")
plotter_ADT2K("xsec_10")
plt.show()

## How does R* behave when 2 identical chain files slowly become more random.
> At what value of R* can we say with confidence that the ML Classifier distinguishes between the 2 chains as good as random guessing.

In [ ]:
#this code is for all_fixed_AdT2K: the 2 files are identical
# NEXT STEP: add more folders that have increasing levels of randomness, use notebook to infer 
path_to_identical= Path("../models/demo_files 2/all_fixed_ADT2K/identical/")
path_to_1= Path("../models/demo_files 2/all_fixed_ADT2K/level_1/")
path_to_2= Path("../models/demo_files 2/all_fixed_ADT2K/level_2/")
path_to_3= Path("../models/demo_files 2/all_fixed_ADT2K/level_3/")
path_to_4= Path("../models/demo_files 2/all_fixed_ADT2K/level_4/")
path_to_5= Path("../models/demo_files 2/all_fixed_ADT2K/level_5/")
path_to_6= Path("../models/demo_files 2/all_fixed_ADT2K/level_6/")
path_to_7= Path("../models/demo_files 2/all_fixed_ADT2K/level_7/")

identical=[f"{path_to_identical}/{x}" for x in os.listdir(path_to_identical)] 
level1=[f'{path_to_1}/{y}' for y in os.listdir(path_to_1)]
level2=[f'{path_to_2}/{z}' for z in os.listdir(path_to_2)]
level3=[f'{path_to_3}/{a}' for a in os.listdir(path_to_3)]# these will be used later as args in functions (used to calc the dims of the data)
level4=[f'{path_to_4}/{b}' for b in os.listdir(path_to_4)]
level5=[f'{path_to_5}/{c}' for c in os.listdir(path_to_5)]
level6=[f'{path_to_6}/{b}' for b in os.listdir(path_to_6)]
level7=[f'{path_to_7}/{c}' for c in os.listdir(path_to_7)]




fitting_label = "ID"
paths = [identical, level1, level2, level3, level4,level5, level6, level7]
for i in paths:
    print(i)
# chain_list [for file in folder ChainHandler(input_file, chain_name, verbose=verbse)]: Handler List : Hlist

temp_identical=[]
temp1=[]
temp2=[]
temp3=[]
temp4=[]
temp5=[]
temp6=[]
temp7=[]



for lm in paths:
    for x in lm:
        chain_handler = ChainHandler(x, 'posteriors', verbose='verbose')
            
        # Process all the files
        chain_handler.ignore_plots(["xsec_20", "xsec_19", 'sin2th_12', 'sin2th_23', 'sin2th_13', 'delm2_12', 'delm2_23', 'delta_cp', 'baseline', 'density', 'Ye'])
        chain_handler.add_additional_plots(["xsec_0", "xsec_1", "xsec_2", "xsec_3", "xsec_4", "xsec_5", "xsec_6", "xsec_7", "xsec_8", "xsec_9", "xsec_10", "xsec_11", "xsec_12", "xsec_13",
                                            "xsec_14", "xsec_15", "xsec_16", "xsec_17", "xsec_18", "xsec_21", "xsec_22"])
        #We need to make sure the chain handler knows this exists, passing true means it is looking for some with that exact name
        # chain_handler.add_additional_plots(fitting_label, True)

        # Finally we can do some cuts to get rid of things like burn-in
        chain_handler.add_new_cuts(["LogL<22.5", "step>10000", "delm2_23>0"])
        chain_handler.add_new_cuts(["step>0", "LogL_systematic_xsec_cov<12345"])

        # Last step is  convert the chain data files into a pandas dataframe
        y= chain_handler.convert_ttree_to_array()
        #chain_list_2.append(chain_handler)


        if lm == identical:
            temp_identical.append(chain_handler)
        elif lm == level1:
            temp1.append(chain_handler)
        elif lm == level2:
            temp2.append(chain_handler)
        elif lm == level3:
            temp3.append(chain_handler)
        elif lm == level4:
            temp4.append(chain_handler)
        elif lm == level5:
            temp5.append(chain_handler)
        elif lm == level6:
            temp6.append(chain_handler)
        elif lm == level7:
            temp7.append(chain_handler)


identical_data = temp_identical[0]
identical_data.ttree_array['ID']= 0 #give the first file, containing chain data, the chain ID of 0

level1_data = temp1[0]
level1_data.ttree_array['ID']= 0

level2_data = temp2[0]
level2_data.ttree_array['ID']= 0

level3_data = temp3[0]
level3_data.ttree_array['ID']= 0

level4_data= temp4[0]
level4_data.ttree_array['ID']=0

level5_data= temp5[0]
level5_data.ttree_array['ID']=0

level6_data= temp6[0]
level6_data.ttree_array['ID']=0

level7_data= temp7[0]
level7_data.ttree_array['ID']=0

# double check if the uniform dist is applied by identifying row (chian id =1) and coloumn (i.e xsec_0)
from itertools import chain
concatenated = chain(range(19), range(21, 23))

for id, chain in enumerate(temp_identical[1:], start = 1): #iterate over the rest of the files and give it an acsending order Chain Ids starting from 1
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    #chain.ttree_array[f'xsec_{rand.randint(0,20)}']= np.random.random(size=len(chain.ttree_array))*1.2#on a random selection of parameters, change each value of that entire coloumn from a random float between 0 and 1
identical_data.ttree_array = pd.concat([identical_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp1[1:], start = 1): 
    
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*0.00001)
    level1_data.ttree_array = pd.concat([level1_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp2[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*0.0001)
    level2_data.ttree_array = pd.concat([level2_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp3[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*0.001)
    level3_data.ttree_array = pd.concat([level3_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp4[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*0.01)
    level4_data.ttree_array = pd.concat([level4_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp5[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*0.1)
    level5_data.ttree_array = pd.concat([level5_data.ttree_array, chain.ttree_array])

for id, chain in enumerate(temp6[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}'])))
    level6_data.ttree_array = pd.concat([level6_data.ttree_array, chain.ttree_array])


for id, chain in enumerate(temp7[1:], start = 1): 
    #add ID to chain['id']
    chain.ttree_array['ID'] = id
    for m in range(19):
        chain.ttree_array[f'xsec_{m}']= chain.ttree_array[f'xsec_{m}'].add(np.random.uniform(-1,1,size=len(chain.ttree_array[f'xsec_{m}']))*10)
    level7_data.ttree_array = pd.concat([level7_data.ttree_array, chain.ttree_array])


#test to see if the code has actually randomly scaled the values:
xsec_values = identical_data.ttree_array[identical_data.ttree_array['ID'] == 1]
print(xsec_values)

l1 = level1_data.ttree_array[level1_data.ttree_array['ID'] == 1]['xsec_15']
print(l1)

l4 = level4_data.ttree_array[level4_data.ttree_array['ID'] == 1]['xsec_15']
print(l4)
l7 = level7_data.ttree_array[level7_data.ttree_array['ID'] == 1]['xsec_15']
print(l7)



# First step we need to initialise the factory
path_to_identical.parent.mkdir(parents=True, exist_ok=True)# Make sure the model output directory exists
print(fitting_label)
# The factory produces ML models, we need to pass it the chain handler and the fitting label to get started
ml_factory_id = MLFactory(identical_data, fitting_label, f"{path_to_identical}.pdf")

#hook up paths to MLFactory
path_to_1.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level1 = MLFactory(level1_data, fitting_label, f"{path_to_1}.pdf")

path_to_2.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level2 = MLFactory(level2_data, fitting_label, f"{path_to_2}.pdf")

path_to_3.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level3 = MLFactory(level3_data, fitting_label, f"{path_to_3}.pdf")

path_to_4.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level4 = MLFactory(level4_data, fitting_label, f"{path_to_4}.pdf")

path_to_5.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level5 = MLFactory(level5_data, fitting_label, f"{path_to_5}.pdf")

path_to_6.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level6 = MLFactory(level6_data, fitting_label, f"{path_to_6}.pdf")

path_to_7.parent.mkdir(parents=True, exist_ok=True)
print(fitting_label)
ml_factory_level7 = MLFactory(level7_data, fitting_label, f"{path_to_7}.pdf")


count = (level1_data.ttree_array['ID'] == 0).sum()
print(count)

count2 = count = (level1_data.ttree_array['ID'] == 1).sum()
print(count2)

# make graph better/ increase the y axis but not much, make more points 



### Run the cell below for diagnostics: Confusion Matrix, Loss function plots, etc

In [ ]:


ml_model_id = ml_factory_id.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0257, max_iter=20, verbose=2, max_leaf_nodes=550, 
                                                   l2_regularization= 0.962, max_features=1.0, early_stopping=True, n_iter_no_change= 10) #contains identical data
ml_model_id.set_training_test_set(0.3)
ml_model_id.train_model()
print(f'results for identical:')
print('=========================')
ml_model_id.test_model_class(identical, 'adapt', 't2k', norm= True, model=ml_model_id)





ml_model_1 = ml_factory_level1.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.07020000000000001, max_iter=20, verbose=2, max_leaf_nodes=220, 
                                                   l2_regularization= 0.8250000000000001, max_features=0.5, early_stopping=True, n_iter_no_change= 10)#contains level1 data
ml_model_1.set_training_test_set(0.3)
ml_model_1.train_model()
print(f'results for level 1:')
print('=========================')
ml_model_1.test_model_class(level1, 'adapt', 't2k', norm= False, model=ml_model_1)


ml_model_2 = ml_factory_level2.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0868, max_iter=20, verbose=2, max_leaf_nodes=100, 
                                                   l2_regularization= 0.395, max_features= 0.5, early_stopping=True, n_iter_no_change= 10)
ml_model_2.set_training_test_set(0.3)
ml_model_2.train_model()
print(f'results for level 2:')
print('=========================')
ml_model_2.test_model_class(level2, 'adapt', 't2k', norm= False, model=ml_model_2)

ml_model_3 = ml_factory_level3.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0152, max_iter=20, verbose=2, max_leaf_nodes=100, 
                                                   l2_regularization= 0.87, max_features= 0.5, early_stopping=True, n_iter_no_change= 10)
ml_model_3.set_training_test_set(0.3)
ml_model_3.train_model()
print(f'results for level 3:')
print('=========================')
ml_model_3.test_model_class(level3, 'adapt', 't2k', norm= False, model=ml_model_3)


ml_model_4 = ml_factory_level4.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.09000000000000001, max_iter=20, verbose=2, max_leaf_nodes=690, 
                                                   l2_regularization= 0.08700000000000001, max_features=0.4, early_stopping=True, n_iter_no_change= 10)
ml_model_4.set_training_test_set(0.3)
ml_model_4.train_model()
print(f'results for level 4:')
print('=========================')

ml_model_4.test_model_class(level4, 'adapt', 't2k', norm= False, model=ml_model_4)




ml_model_5 = ml_factory_level5.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.07940000000000001, max_iter=20, verbose=2, max_leaf_nodes=300, 
                                                   l2_regularization= 0.295, max_features= 0.9, early_stopping=True, n_iter_no_change= 10)
ml_model_5.set_training_test_set(0.3)
ml_model_5.train_model()
print(f'results for level 5:')
print('=========================')
ml_model_5.test_model_class(level5, 'adapt', 't2k', norm= False, model=ml_model_5)


ml_model_6 = ml_factory_level6.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.08370000000000001, max_iter=20, verbose=2, max_leaf_nodes=80, 
                                                   l2_regularization= 0.303, max_features=0.7000000000000001, early_stopping=True, n_iter_no_change= 10)
ml_model_6.set_training_test_set(0.3)
ml_model_6.train_model()
print(f'results for level 6:')
print('=========================')
ml_model_6.test_model_class(level6, 'adapt', 't2k', norm= False, model=ml_model_6) 


ml_model_7 = ml_factory_level7.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.033400000000000006, max_iter=20, verbose=2, max_leaf_nodes=630, 
                                                   l2_regularization= 0.004, max_features= 0.4, early_stopping=True, n_iter_no_change= 10)
ml_model_7.set_training_test_set(0.3)
ml_model_7.train_model()
print(f'results for level 7:')
print('=========================')
ml_model_7.test_model_class(level7, 'adapt', 't2k', norm= False, model=ml_model_7)




In [ ]:

hist = level7_data.ttree_array['xsec_0'][level7_data.ttree_array['ID']==0].hist()
plt.show()
hist2 = level7_data.ttree_array['xsec_0'][level7_data.ttree_array['ID']==1].hist() 
plt.show()

### Calc multiple R* vals for each level and then plot them with error bars

In [ ]:
ml_model_id = ml_factory_id.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0257, max_iter=20, verbose=2, max_leaf_nodes=550, 
                                                   l2_regularization= 0.962, max_features=1.0, early_stopping=True, n_iter_no_change= 10) #contains identical data
ml_model_1 = ml_factory_level1.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.07020000000000001, max_iter=20, verbose=2, max_leaf_nodes=220, 
                                                   l2_regularization= 0.8250000000000001, max_features=0.5, early_stopping=True, n_iter_no_change= 10)#contains level1 data)
ml_model_2 = ml_factory_level2.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0868, max_iter=20, verbose=2, max_leaf_nodes=100, 
                                                   l2_regularization= 0.395, max_features= 0.5, early_stopping=True, n_iter_no_change= 10)

ml_model_3 = ml_factory_level3.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.0152, max_iter=20, verbose=2, max_leaf_nodes=100, 
                                                   l2_regularization= 0.87, max_features= 0.5, early_stopping=True, n_iter_no_change= 10)
ml_model_4 = ml_factory_level4.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.09000000000000001, max_iter=20, verbose=2, max_leaf_nodes=690, 
                                                   l2_regularization= 0.08700000000000001, max_features=0.4, early_stopping=True, n_iter_no_change= 10)
ml_model_5 = ml_factory_level5.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.07940000000000001, max_iter=20, verbose=2, max_leaf_nodes=300, 
                                                   l2_regularization= 0.295, max_features= 0.9, early_stopping=True, n_iter_no_change= 10)
ml_model_6 = ml_factory_level6.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.08370000000000001, max_iter=20, verbose=2, max_leaf_nodes=80, 
                                                   l2_regularization= 0.303, max_features=0.7000000000000001, early_stopping=True, n_iter_no_change= 10)
ml_model_7 = ml_factory_level7.make_interface('SciKit', 'histboostclass', loss='log_loss', learning_rate= 0.033400000000000006, max_iter=20, verbose=2, max_leaf_nodes=630, 
                                                   l2_regularization= 0.004, max_features= 0.4, early_stopping=True, n_iter_no_change= 10)
list_id=[]
list1=[]
list2=[]
list3=[]
list4=[]
list5=[]
list6=[]
list7=[]


datapoints= 3
for i in range(datapoints):
    ml_model_id.set_training_test_set(0.3)
    ml_model_id.train_model()
    y=ml_model_id.getRStar_vals(identical, ml_model_id)
    list_id.append(y)

for i in range(datapoints):
    ml_model_1.set_training_test_set(0.3)
    ml_model_1.train_model()
    y=ml_model_1.getRStar_vals(level1, ml_model_1)
    list1.append(y)

for i in range(datapoints):
    ml_model_2.set_training_test_set(0.3)
    ml_model_2.train_model()
    y=ml_model_2.getRStar_vals(level2, ml_model_2)
    list2.append(y)

for i in range(datapoints):
    ml_model_3.set_training_test_set(0.3)
    ml_model_3.train_model()
    y=ml_model_3.getRStar_vals(level3, ml_model_3)
    list3.append(y)

for i in range(datapoints):
    ml_model_4.set_training_test_set(0.3)
    ml_model_4.train_model()
    y=ml_model_4.getRStar_vals(level4, ml_model_4)
    list4.append(y)

for i in range(datapoints):
    ml_model_5.set_training_test_set(0.3)
    ml_model_5.train_model()
    y=ml_model_5.getRStar_vals(level5, ml_model_5)
    list5.append(y)

for i in range(datapoints):
    ml_model_6.set_training_test_set(0.3)
    ml_model_6.train_model()
    y=ml_model_6.getRStar_vals(level6, ml_model_6)
    list6.append(y)
    

for i in range(datapoints):
    ml_model_7.set_training_test_set(0.3)
    ml_model_7.train_model()
    y=ml_model_7.getRStar_vals(identical, ml_model_7)
    list7.append(y)





In [ ]:

lists=[list_id, list1, list2, list3, list4, list5, list6, list7]
concat= list_id + list1 + list2 + list3 + list4 + list5 + list6 +list7

means = np.array([np.mean(vals) for vals in lists])
mins  = np.array([np.min(vals)  for vals in lists])
maxs  = np.array([np.max(vals)  for vals in lists])

lower_err = means - mins
upper_err = maxs  - means
yerr = np.vstack((lower_err, upper_err))


x = np.arange(len(lists)) + 2   

plt.errorbar(x, means,
             yerr=yerr,
             fmt='o',        
             capsize=5,      s
             elinewidth=1.5,
             markeredgewidth=1)



datapoints = 2
def add_jitter(x, scale=0.2): #scales x-datapoints so that they don't overlap.
    return x + np.random.uniform(-scale, scale, size=len(x))

for xi, vals in zip(x, lists):
    jittered =    jittered_x = add_jitter([xi]*len(vals), scale=0.1)
    plt.scatter(jittered, vals, alpha=0.3, s=30)

plt.xticks([2, 3, 4, 5, 6, 7, 8, 9], ['identical','level 1', 'level 2', 'level 3', 'level 4', 'level 5', 'level 6', 'level 7'])

ymin, ymax = min(concat), max(concat)
pad    = (ymax - ymin) * 0.2
plt.ylim(ymin - pad, ymax + pad)
plt.ylim(min(list1)-0.001, max(list3)+0.001)

plt.gca().yaxis.set_major_locator(plt.MaxNLocator(5))   

plt.title('Sensitivity of R* to randomess')
plt.xlabel('Randomness')
plt.ylabel('R* Value')

plt.show()

labels = ['identical','level 1', 'level 2', 'level 3', 'level 4', 'level 5', 'level 6', 'level 7']

df = pd.DataFrame({
    'labels': labels,
    'Mean': means,
    'Min': mins,
    'Max': maxs,
    'Lower Error': lower_err,
    'Upper Error': upper_err
})

print(df.to_string(index=False))


## Optimising Hyperparameters: RandomizedSearchCV
Given a dict input containing a distribution of hyperparams you want to optimise, and the estimator, it will randomly
draw from the distribution and train the model using it. Calling .best_params_ will output the values of the hyperparams that gave the best result.



+ <b><u>L2_regularisation</u></b>: This essentially applies a penalty (cuts leaves) that are deemed to be useless, this can help reduce over-fitting; increasing the value increases the penalty, the usual values are. from 0.1 to 1, anything from 1 to 10 can lead to under-fitting.

+ <b><u>max_iter</u></b>: changing this changes the actual number of iterations that the model makes

I would want to increase the max_iter value when I actually want to get optimised results, but this should be done in the background 

In [ ]:
# add more hyperparams to optimise

dist={
        'learning_rate': np.arange(0.0001, 0.1, 0.0001 ),
        'max_leaf_nodes': [i for i in range(20, 700, 10)],
        'max_features': np.arange(0.1, 1, 0.1),
        'l2_regularization': np.arange(0.001, 1, 0.001), 
        'max_iter': [20] 
    }

models = [ml_model_id, ml_model_1, ml_model_2, ml_model_3, ml_model_4, ml_model_5, ml_model_6, ml_model_7]


for model in models:
    clf = RandomizedSearchCV(estimator=model.model, param_distributions=dist, cv=5, n_iter=2)
    search = clf.fit(model.scale_data(model._training_data), model.scale_labels(model._training_labels))
    print(f"Best params for {model}: {search.best_params_}")


### Comparing R* values per experiment and per MCMC-type
I want to compare the R* values between T2K and NOvA in Adaptive folder and Non-Adaptive MCMC; maybe also compare the entire folders (aka AD versus Non-AD). Maybe I plot multiple R* vals from per exp and then plot 



In [ ]:
# have to run .test_model_class() on each model before running this
datapoints= 5

T2K_nonadapt= np.random.uniform(1.0, 1.5, 6)#ml_model_NADT2K.getRStar_vals(NAD_T2K, datapoints)
Nova_nonadapt= np.random.uniform(1.0, 1.5, 6)#ml_model_NADT2K.getRStar_vals(NAD_Nova, datapoints)
T2K_adapt=  np.random.uniform(1.0, 1.5, 6)#ml_model_NADT2K.getRStar_vals(AD_T2K, datapoints)
Nova_nonadapt= np.random.uniform(1.0, 1.5, 6)#ml_model_NADT2K.getRStar_vals(AD_Nova, datapoints)



params= {'MCMC-type':['T2K_nonadapt', 'Nova_nonadapt', 'T2K_adapt', 'Nova_nonadapt'],
         'R* Value': [T2K_nonadapt, Nova_nonadapt, T2K_adapt, Nova_nonadapt ]

}

df= pd.DataFrame(params)
print(df)


### Stochasticity of R*

The code below:
 
 - Calculates N values of R* (datapoints = N ), for each file (each file should contain a different number of chains)
 
 - Plots it on the same axis: R* value versus Number of Chains. 
 
 - A jitter-like function was used to offset datapoints by a small amount w/r to x-axis so that points can be distinguishable. 